### ***Criação de Tabelas***

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
conn = sqlite3.connect('aula1sql.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS tb_dados_pessoais (
    id INTEGER,
    nome TEXT,
    idade INTEGER,
    cidade TEXT
    )
''')

conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect('aula1sql.db')
cursor = conn.cursor()

dados = [
    (136, 'João', 25, 'São Paulo'),
    (125, 'Maria', 30, 'Rio de Janeiro'),
    (148, 'Pedro', 28, 'Belo Horizonte'),
    (456, 'Renata', 28, 'Curitiba'),
    (345, 'Marcela', 25, 'São Paulo'),
    (254, 'Ronaldo', 26, 'Curitiba'),
    (178, 'Lucas', 21, 'São Paulo'),
    (365, 'Mateus', 23, 'São Paulo'),
    (145, 'Rebecca', 32, 'Salvador'),
    (258, 'Cassandra', 26, 'São Paulo')
]

cursor.executemany('INSERT INTO tb_dados_pessoais VALUES  (?, ?, ?, ?)', dados)
conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect('aulasql.db')
query = """
SELECT
*
FROM
tb_dados_pessoais
"""
df = pd.read_sql(query, conn)
df

,id,nome,idade,cidade
0,136,João,25,São Paulo
1,125,Maria,30,Rio de Janeiro
2,148,Pedro,28,Belo Horizonte
3,456,Renata,28,Curitiba
4,345,Marcela,25,São Paulo
...,...,...,...,...
65,254,Ronaldo,26,Curitiba
66,178,Lucas,21,São Paulo
67,365,Mateus,23,São Paulo
68,145,Rebecca,32,Salvador


### ***Treinando Join***

In [ ]:
df_funcionarios = pd.read_csv("SQL_TB_FUNCIONARIOS.csv", delimiter=';')

In [ ]:
# Criar uma conexão com um banco de dados SQLite em memória
conn = sqlite3.connect('aula1sql.db')
# Carregar o DataFrame no banco de dados SQLite
df_funcionarios.to_sql('tb_funcionarios', conn, index=False, if_exists='replace')

10

In [ ]:
# Função para executar consultas SQL e retornar o resultado como um DataFrame
def run_query(query):
    return pd.read_sql_query(query, conn)

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
*
FROM tb_funcionarios
INNER JOIN tb_dados_pessoais
ON tb_funcionarios.ID = tb_dados_pessoais.ID
"""

df = pd.read_sql(query, conn)
df

,ID,ÁREA,CARGO,SALÁRIO,TEMPO_NA_EMPRESA,id,nome,idade,cidade
0,136,TI,JUNIOR,2500,0,136,João,25,São Paulo
1,125,TI,SENIOR,6000,1,125,Maria,30,Rio de Janeiro
2,148,TI,SENIOR,6000,2,148,Pedro,28,Belo Horizonte
3,456,ADM,PLENO,4500,2,456,Renata,28,Curitiba
4,345,ADM,PLENO,4600,4,345,Marcela,25,São Paulo
5,254,ADM,SENIOR,5600,5,254,Ronaldo,26,Curitiba
6,178,ADM,JUNIOR,2200,0,178,Lucas,21,São Paulo
7,365,MARKETING,JUNIOR,2300,1,365,Mateus,23,São Paulo
8,145,MARKETING,SENIOR,5500,3,145,Rebecca,32,Salvador
9,258,MARKETING,PLENO,4600,2,258,Cassandra,26,São Paulo


Selecionar apenas as colunas ta tabela de dados pessoais:

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
tb_dados_pessoais.*
FROM tb_funcionarios
LEFT JOIN tb_dados_pessoais
ON tb_funcionarios.ID = tb_dados_pessoais.ID
"""

df = pd.read_sql(query, conn)
df

,id,nome,idade,cidade
0,136,João,25,São Paulo
1,125,Maria,30,Rio de Janeiro
2,148,Pedro,28,Belo Horizonte
3,456,Renata,28,Curitiba
4,345,Marcela,25,São Paulo
5,254,Ronaldo,26,Curitiba
6,178,Lucas,21,São Paulo
7,365,Mateus,23,São Paulo
8,145,Rebecca,32,Salvador
9,258,Cassandra,26,São Paulo


Selecionar apenas as colunas ta tabela de dados pessoais + área da tabela funcionário:

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
tb_dados_pessoais.*,
tb_funcionarios.ÁREA
FROM tb_funcionarios
LEFT JOIN tb_dados_pessoais
ON tb_funcionarios.ID = tb_dados_pessoais.ID
"""

df = pd.read_sql(query, conn)
df

,id,nome,idade,cidade,ÁREA
0,136,João,25,São Paulo,TI
1,125,Maria,30,Rio de Janeiro,TI
2,148,Pedro,28,Belo Horizonte,TI
3,456,Renata,28,Curitiba,ADM
4,345,Marcela,25,São Paulo,ADM
5,254,Ronaldo,26,Curitiba,ADM
6,178,Lucas,21,São Paulo,ADM
7,365,Mateus,23,São Paulo,MARKETING
8,145,Rebecca,32,Salvador,MARKETING
9,258,Cassandra,26,São Paulo,MARKETING


Renomeando a tabela para redução de tempo:

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
A.ÁREA,
B.NOME
FROM tb_funcionarios AS A
INNER JOIN tb_dados_pessoais as B
ON A.ID = B.ID
"""

df = pd.read_sql(query, conn)
df

,ÁREA,nome
0,TI,João
1,TI,Maria
2,TI,Pedro
3,ADM,Renata
4,ADM,Marcela
5,ADM,Ronaldo
6,ADM,Lucas
7,MARKETING,Mateus
8,MARKETING,Rebecca
9,MARKETING,Cassandra


### ***Produzindo Queries Completas***

***1. Vamos montar uma consulta onde trazemos a média da idade dos nossos funcionários por Área.***

**Interpretação**

Colunas: Área, Idade

Filtros: Não

Agrupamento: Sim, Área

Ordenação: Não

Distintos: Não

Origem: tb_funcionários e tb_dados_pessoais

Join: Sim, inner

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
A.ÁREA,
AVG(B.IDADE) AS MEDIA_IDADE
FROM tb_funcionarios AS A
INNER JOIN tb_dados_pessoais as B
ON A.ID = B.ID
GROUP BY 1
"""

df = pd.read_sql(query, conn)
df

,ÁREA,MEDIA_IDADE
0,ADM,25.000000
1,MARKETING,27.000000
2,TI,27.666667


***2. Vamos montar uma consulta onde trazemos o nome dos 3 funcionários com maior salário da empresa e seus respectivos salários ordenados do maior ao menor.***

**Interpretação**

Colunas: Nome, Salário

Filtros: Não

Agrupamento: Não

Ordenação: Sim

Distintos: Não

Origem: tb_funcionários e tb_dados_pessoais

Join: Sim, inner

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
B.NOME,
A.SALÁRIO
FROM tb_funcionarios AS A
INNER JOIN tb_dados_pessoais as B
ON A.ID = B.ID
ORDER BY 2 DESC
LIMIT 3
"""

df = pd.read_sql(query, conn)
df

,nome,SALÁRIO
0,Maria,6000
1,Pedro,6000
2,Ronaldo,5600


***3. Vamos montar uma consulta onde a média das idades por cidade e também a média de salários, além de trazer o nome das cidades.***

**Interpretação**

Colunas: Idade, cidade e salário

Filtros: Não

Agrupamento: Sim, cidade

Ordenação: Não

Distintos: Não

Origem: tb_funcionários e tb_dados_pessoais

Join: Sim, inner

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
B.Cidade,
avg(A.SALÁRIO) as MEDIA_SALARIO,
AVG(IDADE) AS MEDIA_IDADE
FROM tb_funcionarios AS A
INNER JOIN tb_dados_pessoais as B
ON A.ID = B.ID
GROUP BY 1
"""

df = pd.read_sql(query, conn)
df

,cidade,MEDIA_SALARIO,MEDIA_IDADE
0,Belo Horizonte,6000.0,28.0
1,Curitiba,5050.0,27.0
2,Rio de Janeiro,6000.0,30.0
3,Salvador,5500.0,32.0
4,São Paulo,3240.0,24.0


***4. Vamos montar uma consulta trazemos todas informações possíveis de ambas tabelas apenas dos nossos funcionários de TI, ordenando por salário e trazendo a coluna Nome em letra maiuscula.***

**Interpretação**

Colunas:Todas

Filtros: Sim, TI

Agrupamento: Não

Ordenação: Sim, salário

Distintos: Não

Origem: tb_funcionários e tb_dados_pessoais

Join: Sim, inner

In [ ]:
conn = sqlite3.connect('aula1sql.db')
query = """
SELECT
A.*,
UPPER(B.NOME) AS NOME,
B.IDADE,
B.CIDADE
FROM tb_funcionarios AS A
INNER JOIN tb_dados_pessoais as B
ON A.ID = B.ID
WHERE ÁREA IN ('TI')
ORDER BY SALÁRIO DESC
"""

df = pd.read_sql(query, conn)
df

,ID,ÁREA,CARGO,SALÁRIO,TEMPO_NA_EMPRESA,NOME,idade,cidade
0,125,TI,SENIOR,6000,1,MARIA,30,Rio de Janeiro
1,148,TI,SENIOR,6000,2,PEDRO,28,Belo Horizonte
2,136,TI,JUNIOR,2500,0,JOãO,25,São Paulo
